# Construction of mixed-dimensional grids

In [1]:
import numpy as np
import porepy as pp

/home/mok/.venv/lib/python3.12/site-packages/porepy/numerics/nonlinear/nonlinear_solvers.py:14: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import trange  # type: ignore


In [2]:
# The fractures are specified by their vertices, which are 3 x num_pts arrays
f_1 = pp.PlaneFracture(np.array([[0, 2, 2.5, 0],
                                 [0, 0, 1, 1],
                                 [0, 0, 1, 1]]))
f_2 = pp.PlaneFracture(np.array([[1, 1, 1, 1],
                                 [-1, 2, 2, -1],
                                 [-1, -1, 2, 2]]))

f_3 = pp.PlaneFracture(np.array([[-1, 2, 2, -1],
                                [1, 1, 1, 1],
                                 [-1, -1, 2, 2]]))

# Specify the fracture center
center = np.array([0.1, 0.3, 0.2])
# The minor and major axis
major_axis = 1.5
minor_axis = 0.5

# Rotate the major axis around the center.
# Note that the angle is measured in radians
major_axis_angle = np.pi/6

# So far, the fracture is located in the xy-plane. To define the incline, specify the strike angle, and the dip angle.
# Note that the dip rotation is carried out after the major_axis rotation (recall rotations are non-commutative).
strike_angle = -np.pi/3
dip_angle = -np.pi/3

# Finally, the number of points used to approximate the ellipsis.
# This is the only optional parameter; if not specified, 16 points will be used.
num_pt = 12
f_4 = pp.create_elliptic_fracture(center, major_axis, minor_axis, major_axis_angle, strike_angle, dip_angle, num_points=num_pt)


fractures = [f_1, f_2, f_3, f_4]

# Also define the domain
bounding_box = {'xmin': -1, 'xmax': 3, 'ymin': -2, 'ymax': 3, 'zmin': -1.5, 'zmax': 3}
domain = pp.Domain(bounding_box=bounding_box)

# Define a 3d FractureNetwork, similar to the 2d one
network = pp.create_fracture_network(fractures)

# Generate the mixed-dimensional mesh
mesh_args = {'cell_size_boundary': 1.0, 'cell_size_fracture': 0.5, 'cell_size_min': 0.2}
mdg = pp.create_mdg("simplex", mesh_args, network)

In [3]:
pp.Exporter(mdg, 'mixed_dimensional_grid').write_vtu()